phase 6: advanced optimization & model stacking

merging external climate features, engineering cyclical/spatial features, and stacking models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
print('--- loading and merging datasets ---')
train = pd.read_csv('../data/raw/Train.csv')
test = pd.read_csv('../data/raw/Test.csv')
climate = pd.read_csv('../references/climate_features.csv')

# drop deathdate from climate as we will extract it from train
if 'deathdate' in climate.columns:
    climate = climate.drop(columns=['deathdate'])

train_merged = train.merge(climate, on='ID', how='left')
test_merged = test.merge(climate, on='ID', how='left')

print(f'train merged shape: {train_merged.shape}')
print(f'test merged shape: {test_merged.shape}')

In [ ]:
print('--- advanced feature engineering ---')

def engineer_features(df):
    df = df.copy()
    
    # 1. date features (cyclical month)
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['month'] = df['deathdate'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    
    # 2. cross features
    df['temp_range'] = df['max_temperature'] - df['min_temperature']
    df['is_infant'] = (df['age'] <= 5).astype(int)
    
    # drop raw date/string columns
    cols_to_drop = ['ID', 'deathdate', 'location', 'month']
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

train_fe = engineer_features(train_merged)
test_fe = engineer_features(test_merged)

# 3. spatial clustering (k-means)
print('--- applying spatial clustering ---')
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

# fit only on train to prevent leakage
spatial_coords_train = train_fe[['latitude', 'longitude']]
spatial_coords_test = test_fe[['latitude', 'longitude']]

train_fe['spatial_cluster'] = kmeans.fit_predict(spatial_coords_train)
test_fe['spatial_cluster'] = kmeans.predict(spatial_coords_test)

# one-hot encode spatial_cluster, gender, zone
train_fe = pd.get_dummies(train_fe, columns=['spatial_cluster', 'gender', 'zone'], drop_first=True)
test_fe = pd.get_dummies(test_fe, columns=['spatial_cluster', 'gender', 'zone'], drop_first=True)

# align test columns with train
X = train_fe.drop(columns=['is_climate_sensitive'])
y = train_fe['is_climate_sensitive']
X_test_final = test_fe.reindex(columns=X.columns, fill_value=0)

print(f'final features count: {X.shape[1]}')
print(f'test features count aligned: {X_test_final.shape[1]}')

In [ ]:
print('--- training stacking ensemble with 10-fold cv ---')

scale_weight = (y == 0).sum() / (y == 1).sum()

# base models (diverse architectures and hyperparams)
xgb = XGBClassifier(scale_pos_weight=scale_weight, max_depth=4, learning_rate=0.05, n_estimators=200, random_state=42, n_jobs=-1, eval_metric='logloss')
rf = RandomForestClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=10, n_estimators=200, random_state=42, n_jobs=-1)
xgb_deep = XGBClassifier(scale_pos_weight=scale_weight, max_depth=8, learning_rate=0.01, subsample=0.7, n_estimators=300, random_state=42, n_jobs=-1, eval_metric='logloss')

# meta model (logistic regression to combine predictions)
meta = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)

stacking = StackingClassifier(
    estimators=[('xgb', xgb), ('rf', rf), ('xgb_deep', xgb_deep)],
    final_estimator=meta,
    cv=5, # internal cv for the meta-model
    n_jobs=-1
)

# 10-fold cv evaluation on the entire stack
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds = cross_val_predict(stacking, X, y, cv=cv, method='predict')
oof_probs = cross_val_predict(stacking, X, y, cv=cv, method='predict_proba')[:, 1]

cv_f1 = f1_score(y, oof_preds)
cv_auc = roc_auc_score(y, oof_probs)

print(f'10-fold cv f1 score: {cv_f1:.4f}')
print(f'10-fold cv roc-auc:  {cv_auc:.4f}')

if cv_f1 >= 0.90 and cv_auc >= 0.90:
    print('\nsuccess: target 0.90 achieved! the external data worked.')
else:
    print('\nwe improved, but still short of 0.90. check the metrics to see the jump!')

In [ ]:
print('--- generating final submission ---')
# train on full dataset
stacking.fit(X, y)

# predict on test set
test_preds = stacking.predict(X_test_final)
test_probs = stacking.predict_proba(X_test_final)[:, 1]

# create submission
sub = test[['ID']].copy()
sub['TargetF1'] = test_preds
sub['TargetRAUC'] = test_probs

sub.to_csv('../data/processed/SampleSubmission_Phase6.csv', index=False)
print('saved to data/processed/SampleSubmission_Phase6.csv')